Hello... this is the jupyter notebook for **gene*Flow***, to guide the user to predict perturbation effects of new genes. We will also explain some of the code here. For our [paper](https://github.com/MikelGarciaData/DL_VCC/blob/main/Deep_Learning_project.pdf) the main file is [train_m_flow.py](https://github.com/MikelGarciaData/DL_VCC/blob/main/geneFlow/train_m_flow.py).

# Setting up Data

You can download the zip file(4 GB) for this project from https://dtudk-my.sharepoint.com/:u:/g/personal/s225191_dtu_dk/EVEOd_TBVgBBpCjmuHGtTkEBRNq_IvEiY-wK4n-MgB4IKQ?e=QdVElI.

You can also download the 15 GB adata file directly.

In [ ]:
import sys
sys.path.append(".")   # add DL_VCC to the import path

In [ ]:
import os
os.makedirs("data", exist_ok=True)

In [ ]:
!curl -L -o data/adata_raw.h5ad "https://huggingface.co/datasets/cyrilzakka/arc-institute-virtual-cell-dataset/resolve/main/adata_Training.h5ad"

# scVI embeddings

We used [scVI-tools](https://docs.scvi-tools.org/en/stable/tutorials/notebooks/quick_start/api_overview.html) to embed the 18,080 genes in a 50-dimensional latent space.

We first tried using the Arc Institute’s [SE](https://github.com/ArcInstitute/state) model, which produces 2,080-dimensional embeddings. However, when we trained our model using these embeddings, computing the Flow Matching loss became difficult — the loss dropped to approximately 0 after the first training round, including the validation loss.

We believe this happened because the embeddings were still effectively 2,080-dimensional, and even after dimensionality reduction, the high dimensionality led to a curse-of-dimensionality effect. As a result, the model struggled to  distinguish between cell states. For this reason, we switched to using scVI, which provides a more manageable 50-dimensional latent space.

We need to create a venv.

```bash
python3 -m venv geneflow_env
source geneflow_env/bin/activate
```

In [ ]:
#install scvi-tools with cuda support
%pip install scvi-tools[cuda]
# Install required packages for geneFlow
%pip3 install torch torchvision
%pip3 scanpy anndata tqdm collections scikit-learn argparse numpy

You can find the code we used [here](https://github.com/MikelGarciaData/DL_VCC/blob/main/helpers/scvi_embed.py).

In [ ]:
from helpers.scvi_embed import run_scvi_embedding

In [ ]:
run_scvi_embedding(
    raw_data_path="data/adata_raw.h5ad",
    embedded_data_path="data/adata_scvi_50.h5ad",
    n_latent=50
)

# geneFlow

Now that we have the embbeded data we can train our model, here we will use the file [train_m_flow.py](https://github.com/MikelGarciaData/DL_VCC/blob/main/geneFlow/train_m_flow.py). We implemented seeding in this file, so that the data can be replicated.

But first we need to install the dependencies. 

## Gene embeddings

The dataset we used has **149** human genes, which are down regulated to test for perturbation effects. Our main goal is to check if the model can predict the perturbation effects of new genes, not used in the training set. For this we need to encode the identity of the gene in a vector, to allow the model learn the functioning of the gene and predict effects of similar genes.

We used [genePT](https://github.com/yiqunchen/GenePT) to get the embeddings of our genes and stored them in [149_genePT.txt](https://github.com/MikelGarciaData/DL_VCC/blob/main/geneFlow/149_genePT.txt). In short GenePT creates gene embeddings by converting each gene’s NCBI text description into a numeric vector using GPT-3.5. Which capture gene function and effects from literature. [Studies](https://www.biorxiv.org/content/10.1101/2025.01.29.635607v1) have found GenePT to give high performing embeddings.

## Training

We train the model with our embedded data. The [train_m_flow.py](https://github.com/MikelGarciaData/DL_VCC/blob/main/geneFlow/train_m_flow.py) first make a training set and a validation set based on the perturbation genes. By default the script chooses 80% of the genes for training. Hence, the cells that are perturbed with the 119 selected genes are used for training. And then validated with the cells perturbed with the 30 genes not from the training set. *To be clear a cell is perturbed with only one gene.* 

The model is trained to predict the velocity (*where should the cell move in the 50 dimensional gene expression latent space*) at a specific time-step so that it is closer to the final perturbed state. We call it predicting the flow of the cell. You can read more about the theory and logic behind the loss calculation in our [paper](https://github.com/MikelGarciaData/DL_VCC/blob/main/Deep_Learning_project.pdf).

The model trains on the whole training set every epoch to give a training loss and a validation loss. Once the validation loss stops decreasing for 10 epochs, the training is stopped i.e. early stopping. We then save the best model.

## Predictions

Once we have trained the model on the training set, we predict the final perturbed state of the cell. We use the genes from the validation set to make predictions on the control cells chosen at random. We predict the same amount cells as in the true dataset.

To get the final state we use the function `ode_solve_trajectory()`, which solves the trajectory from the control cell to the perturbed state based on the predicted value of the velocity at each time-step. The final state is calculated by integrating the predictions.

The function saves the snapshots of the predicted gene expresion at t=0, 0.25, 0.5, 0.75, 1.0 to see the trajectory. The t=1.0 is the final perturbed state.

## Running **gene*Flow***

`train_m_flow.py` **Arguments**



| Argument | Type | Default | Description |
|----------|------|---------|-------------|
| `--adata_path` | str | Required | Path to the input `.h5ad` file containing the single-cell data. |
| `--gene_vec_path` | str | Required | Path to the file containing gene embeddings. |
| `--save_dir` | str | `./checkpoints` | Directory where model checkpoints, configuration, loss history, and predictions will be saved. |
| `--emb_key` | str | `X_scvi` | Key in `adata.obsm` that contains the embeddings to be used as input. |
| `--gene_col` | str | `target_gene` | Column in `adata.obs` specifying the gene associated with each cell. |
| `--control_label` | str | `non-targeting` | Label identifying control cells in `adata.obs[gene_col]`. |
| `--epochs` | int | `50` | Number of training epochs. |
| `--batch_size` | int | `128` | Number of samples per training batch. |
| `--val_split` | float | `0.2` | Fraction of perturbed genes to use for validation. |
| `--hidden_dim` | int | `128` | Hidden dimension size for the neural network. |
| `--num_layers` | int | `6` | Number of residual blocks in the VectorFlowNet model. |
| `--seed` | int | `42` | Random seed for reproducibility. |
| `--pred_steps` | int | `20` | Number of ODE solver steps during trajectory prediction (should be divisible by 4). |

---



In [ ]:
%run geneFlow/train_m_flow.py \
    --adata_path data/adata_scvi_50.h5ad \
    --gene_vec_path geneFlow/149_genePT.txt \
    --save_dir ./results_experiment_200_epoch_256_8_notebook \
    --emb_key X_scvi \
    --gene_col target_gene \
    --control_label non-targeting \
    --epochs 200 \
    --pred_steps 32 \
    --hidden_dim 256 \
    --num_layers 8

The command trains the model on the embedded data, saves the best model, training/validation loss, genes used in the training/validation sets, and then predicts the final gene expresion of the perturbation genes from the validation set in the `save_dir`.

After running [train_m_flow.py](https://github.com/MikelGarciaData/DL_VCC/blob/main/geneFlow/train_m_flow.py), we visualise the predictions of the model by using the code from [plot_traj.py](https://github.com/MikelGarciaData/DL_VCC/blob/main/geneFlow/plot_traj.py)

In [ ]:
results_dir = "./results_experiment_200_epoch_256_8_notebook"

In [ ]:
import scanpy as sc
import os
import matplotlib.pyplot as plt

time_points = [0.00, 0.25, 0.50, 0.75, 1.00]
fig, axes = plt.subplots(1, 5, figsize=(25, 5))

print("Generating UMAP plots...")
for i, t in enumerate(time_points):
    fname = f"predictions_t_{t:.2f}.h5ad"
    path = os.path.join(results_dir, fname)
    
    # Get the current axis (subplot)
    ax = axes[i]
    
    if os.path.exists(path):
        print(f"Processing t={t}...")
        
        # Load Data
        adata_t = sc.read_h5ad(path)
        
        # Compute UMAP (Independent for each timepoint)
        sc.pp.neighbors(adata_t, use_rep='X', n_neighbors=30)
        sc.tl.umap(adata_t, random_state=42)

        # Plot
        sc.pl.umap(
            adata_t,
            color="condition",
            size=2,
            ax=ax,
            show=False, 
            title=f"Time: {t}",
            legend_loc='right margin' if i == 4 else None 
        )
    else:
        print(f"Warning: File {fname} not found.")
        ax.text(0.5, 0.5, "Missing File", ha='center')
        ax.axis('off')

plt.tight_layout()
plt.show()